# Two-layer QG testbed: 4DVarNet vs 4DVar, and the Lyapunov view of the gain

Whole pipeline in one notebook (same code as the hydra xps `qg2l_*`, see `contrib/qg2l/README.md`):

1. truth run + nadir / SWOT observations, figures and animation
2. 4DVarNet training / test (state = psi_1, psi_2; SSH observations only)
3. cycled 4DVar, strong and weak constraint (imperfect low-resolution model, autograd adjoint)
4. comparison plots and scores
5. projection of the reconstruction errors on the BLV / CLV of the DA model

`SIZE = "small"` (default) runs everything on a 32x32, 500 km box in minutes on a GPU (longer on CPU);
`SIZE = "full"` is the 64x64, 1000 km set-up of the hydra configs (GPU, hours).

In [ ]:
import os, sys, json
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "main.py").exists():
    REPO = REPO.parent
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import xarray as xr
from IPython.display import Image, Video, display

from contrib.qg2l import simulate, viz, metrics, obs as O
from contrib.qg2l.qgmodel import QG2L

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 0. Configuration

In [ ]:
SIZE = "small"   # "small" | "full"

if SIZE == "small":
    CFG = dict(L=5e5, nx=32, nx_truth=64, spinup_days=300,
               splits=dict(train=[0, 1095], val=[1095, 1185], test=[1215, 1305], total=1395),
               out_dir="data/qg2l_nb_small")
    # window >= 2-3 revisit times (median revisit ~ 7.6 d), full resolution (downsamp=None)
    NET = dict(n_time=21, epochs=120, n_step=15, batch_size=16, hidden_prior=64, hidden_grad=96, aug_factor=5,
               downsamp=None)
    VAR = dict(window_days=7, n_iter=40, spinup_days=30, b_scale=0.5, q_scale=0.05)
    LYAP = dict(k=192, n_fwd=30, n_bwd=60)
else:
    CFG = dict(L=1e6, nx=64, nx_truth=128, spinup_days=1095,
               splits=dict(train=[0, 1095], val=[1095, 1277], test=[1307, 1672], total=1762),
               out_dir="data/qg2l")
    NET = dict(n_time=21, epochs=150, n_step=15, batch_size=16, hidden_prior=64, hidden_grad=96, aug_factor=5,
               downsamp=None)
    VAR = dict(window_days=10, n_iter=60, spinup_days=60, b_scale=0.5, q_scale=0.05)
    LYAP = dict(k=512, n_fwd=30, n_bwd=60)

PRESET = "energetic"
OUT = Path(CFG["out_dir"]); REC = OUT / "reconstructions"; FIG = OUT / "figures"
DATA = OUT / f"qg2l_{PRESET}_nx{CFG['nx']}_truth{CFG['nx_truth']}.nc"

# keep the ground-track spacing of the real constellation when the box is smaller
CONSTELLATION = {name: dict(sat, tracks_per_dir=max(1, round(sat["tracks_per_dir"] * CFG["L"] / 1e6)))
                 for name, sat in O.DEFAULT_CONSTELLATION.items()}
CONSTELLATION

## 1. Truth run and observations

In [ ]:
# (re)generate when missing or produced with other splits (e.g. an older version of this notebook)
if DATA.exists() and json.loads(xr.open_dataset(DATA, decode_times=False).attrs["splits"]) != CFG["splits"]:
    print("existing file has other splits -> regenerating"); DATA.unlink()
if not DATA.exists():
    simulate.generate(out_dir=str(OUT), preset=PRESET, nx=CFG["nx"], dt=7200.0, nx_truth=CFG["nx_truth"],
                      dt_truth=3600.0, L=CFG["L"], spinup_days=CFG["spinup_days"], splits=CFG["splits"],
                      constellation=CONSTELLATION, backend="torch", device=DEVICE, animation_days=60)
ds = xr.open_dataset(DATA, decode_times=False).load()
KW = json.loads(ds.attrs["model_kw"])
summary = json.loads(ds.attrs["summary"])
print(json.dumps(summary, indent=1))
print(f"lambda_1 x median revisit = {summary['lambda1_x_revisit']:.2f}")

In [ ]:
for f in ["snapshot_test_day0.png", "spectra.png", "obs_coverage.png", "energy.png"]:
    if (FIG / f).exists():
        display(Image(filename=str(FIG / f)))

In [ ]:
anim = FIG / "truth_obs.mp4"
display(Video(str(anim), embed=True, width=900) if anim.exists() else Image(filename=str(anim.with_suffix(".gif"))))

## 2. 4DVarNet

Same components as `xp=qg2l_4dvarnet`: `src.models.GradSolver` with the bilinear AE prior and the ConvLSTM
gradient model (circular padding), trained with `LitQG2L` (subclass of `Lit4dVarNet`). The test step blends the
overlapping windows and writes `reconstructions/4dvarnet.nc`.

In [ ]:
import pytorch_lightning as pl
import src.models
from contrib.qg2l.data import QG2LDataModule, qg_rec_weight, n_channels
from contrib.qg2l.models import LitQG2L, periodic

pl.seed_everything(333)
LAYERS = [1, 2]
C = n_channels(NET["n_time"], LAYERS)
dm = QG2LDataModule(str(DATA), n_time=NET["n_time"], layers=LAYERS, aug_factor=NET["aug_factor"],
                    dl_kw=dict(batch_size=NET["batch_size"], num_workers=0))

solver = src.models.GradSolver(
    prior_cost=periodic(src.models.BilinAEPriorCost(C, NET["hidden_prior"], downsamp=NET["downsamp"], bilin_quad=False)),
    obs_cost=src.models.BaseObsCost(),
    grad_mod=periodic(src.models.ConvLstmGradModel(C, NET["hidden_grad"], downsamp=NET["downsamp"])),
    n_step=NET["n_step"], lr_grad=1e3,
)

def opt_fn(lit, lr=1e-3):   # same as src.utils.cosanneal_lr_adam
    opt = torch.optim.Adam([
        {"params": lit.solver.grad_mod.parameters(), "lr": lr},
        {"params": lit.solver.obs_cost.parameters(), "lr": lr},
        {"params": lit.solver.prior_cost.parameters(), "lr": lr / 2},
    ])
    return {"optimizer": opt, "lr_scheduler": torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NET["epochs"])}

lit = LitQG2L(solver=solver, rec_weight=qg_rec_weight(NET["n_time"], LAYERS, CFG["nx"]), opt_fn=opt_fn,
              persist_rw=False, export_path=str(REC / "4dvarnet.nc"))
logger = pl.loggers.CSVLogger(str(OUT / "logs"), name="4dvarnet")
trainer = pl.Trainer(max_epochs=NET["epochs"], accelerator="auto", devices=1, inference_mode=False,
                     gradient_clip_val=0.5, logger=logger,
                     callbacks=[pl.callbacks.ModelCheckpoint(monitor="val_mse", save_top_k=1)])
trainer.fit(lit, datamodule=dm)
trainer.test(lit, datamodule=dm, ckpt_path="best")

In [ ]:
hist = pd.read_csv(Path(logger.log_dir) / "metrics.csv").groupby("epoch").mean(numeric_only=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
hist[[c for c in ["train_loss", "val_loss"] if c in hist]].plot(ax=ax[0], logy=True)
hist[[c for c in ["train_mse", "val_mse"] if c in hist]].plot(ax=ax[1], logy=True)
ax[0].set_title("loss"); ax[1].set_title("MSE (x 1e4, physical units)");

## 3. Cycled 4DVar (strong and weak constraint)

Low-resolution QG model (= model error with respect to the high-resolution truth), spectral B from the training
climatology, L-BFGS, adjoint by autograd. `b_scale` / `q_scale` / `window_days` should be tuned on the validation
split (`split="val"`) before the final comparison.

In [ ]:
from contrib.qg2l.fourdvar import run_4dvar

for mode in ["strong", "weak"]:
    out = REC / f"4dvar_{mode}_test.nc"
    if not out.exists():
        run_4dvar(str(DATA), str(out), mode=mode, window_days=VAR["window_days"], b_scale=VAR["b_scale"],
                  q_scale=VAR["q_scale"], n_iter=VAR["n_iter"], spinup_days=VAR["spinup_days"], device=DEVICE)

## 4. Comparison

In [ ]:
RECS = {"4dvarnet": REC / "4dvarnet.nc", "4dvar_strong": REC / "4dvar_strong_test.nc",
        "4dvar_weak": REC / "4dvar_weak_test.nc"}
RECS = {k: v for k, v in RECS.items() if v.exists()}
t0, t1 = CFG["splits"]["test"]
f0g = KW["f0"] / KW["g"]
truth = ds.psi.values[t0:t1].astype(np.float64)
rec = {k: xr.open_dataset(v, decode_times=False).psi.values.astype(np.float64) for k, v in RECS.items()}

pd.DataFrame({k: json.loads(xr.open_dataset(v, decode_times=False).attrs["scores"]) for k, v in RECS.items()}).round(4)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
days = ds.time.values[t0:t1]
for name, r in rec.items():
    for l in range(2):
        ax[l].plot(days, f0g * np.sqrt(((r[:, l] - truth[:, l]) ** 2).mean((-1, -2))), label=name)
ax[0].set_title("RMSE layer 1 (SSH) [m]"); ax[1].set_title("RMSE layer 2 (SSH-equivalent) [m]")
ax[1].set_xlabel("day"); ax[0].legend();

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
dx = KW["L"] / KW["nx"]
for name, r in rec.items():
    for l in range(2):
        wl, score, lam = metrics.psd_score(f0g * r[:, l], f0g * truth[:, l], dx)
        ax[l].semilogx(wl, score, label=f"{name} ({lam:.0f} km)")
for l in range(2):
    ax[l].axhline(0.5, c="gray", lw=0.8); ax[l].set_ylim(-0.2, 1.05); ax[l].invert_xaxis()
    ax[l].set_xlabel("wavelength [km]"); ax[l].set_title(f"PSD score, layer {l+1}"); ax[l].legend()

In [ ]:
day = (t1 - t0) // 2
names = list(rec)
ext = [0, KW["L"] / 1e3, 0, KW["L"] / 1e3]
fig, ax = plt.subplots(2, 2 + 2 * len(names), figsize=(3.2 * (2 + 2 * len(names)), 6.2), constrained_layout=True)
obs = ds.obs_ssh.values[t0 + day]
for l in range(2):
    v = np.abs(f0g * truth[day, l]).max()
    panels = [(f"truth L{l+1}", f0g * truth[day, l], v), ("obs (SSH)" if l == 0 else "", obs if l == 0 else None, v)]
    e_max = max(np.abs(f0g * (rec[n][day, l] - truth[day, l])).max() for n in names)
    for n in names:
        panels.append((f"{n} L{l+1}", f0g * rec[n][day, l], v))
    for n in names:
        panels.append((f"err {n} L{l+1}", f0g * (rec[n][day, l] - truth[day, l]), e_max))
    for a, (title, fld, vm) in zip(ax[l], panels):
        a.set_xticks([]); a.set_yticks([])
        if fld is None:
            a.axis("off"); continue
        a.imshow(fld, origin="lower", extent=ext, cmap="RdBu_r", vmin=-vm, vmax=vm); a.set_title(title, fontsize=9)
fig.suptitle(f"test day {day}");

## 5. Lyapunov diagnostic

BLV / CLV of the low-resolution DA model along the true trajectory (energy norm), and projection of each error on the
unstable / neutral / stable-resolved / residual subspaces. Check `n_unstable < k` in the summary.

In [ ]:
from contrib.qg2l.diag import run_diag

summary_lyap = run_diag(str(DATA), {k: str(v) for k, v in RECS.items()}, str(OUT / "diag"), k=LYAP["k"],
                        n_transient_fwd=LYAP["n_fwd"], n_transient_bwd=LYAP["n_bwd"], device=DEVICE)

In [ ]:
for f in ["lyapunov_spectrum_error_distribution.png", "subspace_summary.png",
          "error_energy_by_subspace.png", "clv_and_error_maps.png"]:
    display(Image(filename=str(OUT / "diag" / f)))
anim = OUT / "diag" / "reconstructions.mp4"
display(Video(str(anim), embed=True, width=1100) if anim.exists() else Image(filename=str(anim.with_suffix(".gif"))))

In [ ]:
s = summary_lyap
print(f"lambda_1 = {s['lambda1_per_day']:.3f} /day, n_unstable = {s['n_unstable']} / k = {s['k']}", s["warning"] or "")
pd.DataFrame(s["methods"]).T.round(4)

In [ ]:
# relative gain of 4DVarNet per subspace (1 - E_4dvarnet / E_4dvar)
pd.DataFrame(s.get(f"gain_vs_{list(s['methods'])[0]}", {})).T.round(3)